# MIMIC-IV DN Transformer Training

This notebook is tailored to the six supplied Parquet files and trains:

- **ClinicalBERT EHR-to-text**
- **Med-BERT-style**
- **TransTab**
- **FT-Transformer**

Default outcome: `primary_icd_dn_label`  
Default early prediction window: **24 hours**

The notebook uses `patient_splits.parquet` as the fixed train/validation/test split and excludes several obvious leakage-prone variables from the default feature set.


In [11]:
%pip install --upgrade ipywidgets jupyter

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------- ----------- 1.6/2.2 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 7.9 MB/s  0:00:00

   ---------------- ----------------------- 2/5 [ipywidgets]
   ---------------- ----------------------- 2/5 [ipywidgets]
   ---------------------------------------- 5/5 [jupyter]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
MIMIC-IV Diabetic Nephropathy (DN) risk prediction
===================================================

Models
------
1. ClinicalBERT (EHR-to-text adaptation using the supplied files)
2. Med-BERT-style longitudinal ICD Transformer
3. TransTab
4. FT-Transformer

Supplied files
--------------
- cohort_labels.parquet
- dx_sequences.parquet
- early_features_24h.parquet
- early_features_48h.parquet
- patient_splits.parquet
- structured_features.parquet

IMPORTANT METHODOLOGICAL NOTES
------------------------------
1. patient_splits.parquet is treated as the authoritative train/validation/test split.
2. Default target = primary_icd_dn_label.
   Alternatives:
       secondary_lab_proxy_label
       strict_sensitivity_label
3. Default prediction window = first 24 hours.
   Change FEATURE_WINDOW to "48h" for a sensitivity experiment.
4. Index-admission diagnosis codes are excluded from the longitudinal sequence to
   reduce target leakage. Only visits BEFORE the index admission are used.
5. Outcome-defining fields (DN labels, max_creatinine, max_bun, ESRD/dialysis flag)
   are NOT used as predictors.
6. ClinicalBERT normally expects clinical language. Because no notes file was
   supplied, this notebook converts the available EHR variables and prior ICD codes
   into patient-level text. Report this as "ClinicalBERT EHR-to-text" rather than
   conventional note-based ClinicalBERT. For a strict ClinicalBERT experiment,
   add MIMIC-IV-Note text available before the prediction time.
"""

In [1]:
# ============================================================
# 0. JUPYTER INSTALLATION
# Run these in separate cells in the mimic_dn Python 3.10 kernel
# ============================================================

%pip install pyarrow pandas numpy scikit-learn
%pip install "transformers<=4.30.0" accelerate
%pip install pytorch-tabular
%pip install git+https://github.com/RyanWangZf/transtab.git

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/RyanWangZf/transtab.git to C:\Users\oluwa\AppData\Local\Temp\pip-req-build-dc0hf3qz
  Resolved https://github.com/RyanWangZf/transtab.git to commit fdb34cf38abda73ee6a741b802fe226cc89ba7b5
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/RyanWangZf/transtab.git 'C:\Users\oluwa\AppData\Local\Temp\pip-req-build-dc0hf3qz'


In [1]:
# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================
from pathlib import Path
from collections import Counter, defaultdict
import copy
import json
import os
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

SEED = 42

# If the notebook is saved in the SAME folder as the six parquet files:
DATA_DIR = Path.cwd()

# Otherwise replace the line above with your Windows folder, for example:
# DATA_DIR = Path(r"C:\Users\oluwa\Documents\MIMIC_DN")

FILES = {
    "labels": DATA_DIR / "cohort_labels.parquet",
    "dx": DATA_DIR / "dx_sequences.parquet",
    "early24": DATA_DIR / "early_features_24h.parquet",
    "early48": DATA_DIR / "early_features_48h.parquet",
    "splits": DATA_DIR / "patient_splits.parquet",
    "structured": DATA_DIR / "structured_features.parquet",
}

ID_COL = "subject_id"
TARGET_COL = "primary_icd_dn_label"
# Other available targets:
# TARGET_COL = "secondary_lab_proxy_label"
# TARGET_COL = "strict_sensitivity_label"

FEATURE_WINDOW = "24h"     # change to "48h" for sensitivity analysis

# Because your current PyTorch installation may be CPU-only, these defaults are conservative.
CLINICALBERT_MAX_LEN = 256
MEDBERT_MAX_LEN = 256

RUN_CLINICALBERT = True
RUN_MEDBERT = True
RUN_TRANSTAB = True
RUN_FTTRANSFORMER = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Device: cpu
PyTorch: 2.13.0+cpu
CUDA available: False


In [2]:
# ============================================================
# 2. LOAD THE SIX PARQUET FILES
# ============================================================
for name, path in FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}\n"
            "Put the notebook in the same folder as the parquet files "
            "or change DATA_DIR."
        )

labels = pd.read_parquet(FILES["labels"])
dx = pd.read_parquet(FILES["dx"])
early24 = pd.read_parquet(FILES["early24"])
early48 = pd.read_parquet(FILES["early48"])
splits = pd.read_parquet(FILES["splits"])
structured = pd.read_parquet(FILES["structured"])

tables = {
    "cohort_labels": labels,
    "dx_sequences": dx,
    "early_features_24h": early24,
    "early_features_48h": early48,
    "patient_splits": splits,
    "structured_features": structured,
}

for name, frame in tables.items():
    print(f"\n{name}: {frame.shape}")
    print(frame.columns.tolist())


cohort_labels: (46312, 21)
['subject_id', 'index_hadm_id', 'index_admittime', 'index_dischtime', 'deathtime', 'admission_type', 'discharge_location', 'race', 'insurance', 'gender', 'age_at_admission', 'n_eligible_admissions', 'type2_only_flag', 'index_los_days', 'primary_icd_dn_label', 'secondary_lab_proxy_label', 'strict_sensitivity_label', 'esrd_or_dialysis_flag', 'max_creatinine', 'max_bun', 'split']

dx_sequences: (596127, 9)
['subject_id', 'hadm_id', 'visit_number', 'age_at_visit', 'days_since_prev_visit', 'icd_code', 'icd_version', 'code_rank_within_visit', 'split']

early_features_24h: (46312, 5)
['subject_id', 'mean_glucose_24h', 'max_creatinine_24h', 'max_bun_24h', 'mean_sbp_24h']

early_features_48h: (46312, 5)
['subject_id', 'mean_glucose_48h', 'max_creatinine_48h', 'max_bun_48h', 'mean_sbp_48h']

patient_splits: (46312, 2)
['subject_id', 'split']

structured_features: (46312, 39)
['subject_id', 'index_hadm_id', 'gender', 'age_at_admission', 'race', 'insurance', 'admission_

In [3]:
# ============================================================
# 3. VALIDATE LABELS AND PREDEFINED PATIENT SPLITS
# ============================================================
required_label_cols = {
    ID_COL,
    "index_hadm_id",
    "primary_icd_dn_label",
    "secondary_lab_proxy_label",
    "strict_sensitivity_label",
}

missing = required_label_cols - set(labels.columns)
if missing:
    raise ValueError(f"cohort_labels.parquet is missing: {missing}")

if TARGET_COL not in labels.columns:
    raise ValueError(f"Target {TARGET_COL!r} not found in cohort_labels.parquet")

if splits[ID_COL].duplicated().any():
    raise ValueError("patient_splits.parquet contains duplicate subject_id values.")

valid_split_names = {"train", "validation", "test"}
observed_splits = set(splits["split"].dropna().astype(str).str.lower().unique())

if not observed_splits.issubset(valid_split_names):
    raise ValueError(
        f"Unexpected split names: {observed_splits}. "
        f"Expected only {valid_split_names}."
    )

# Use patient_splits as authoritative, rather than split copies inside other files.
split_map = splits[[ID_COL, "split"]].copy()
split_map["split"] = split_map["split"].astype(str).str.lower()

label_base = (
    labels[
        [
            ID_COL,
            "index_hadm_id",
            TARGET_COL,
        ]
    ]
    .drop_duplicates(ID_COL)
    .merge(split_map, on=ID_COL, how="inner")
)

label_base[TARGET_COL] = label_base[TARGET_COL].astype(int)

if not set(label_base[TARGET_COL].unique()).issubset({0, 1}):
    raise ValueError(f"{TARGET_COL} must contain only 0 and 1.")

print("\nTarget:", TARGET_COL)
print("\nSplit sizes:")
print(label_base["split"].value_counts())

print("\nDN label distribution by split:")
print(
    label_base.groupby("split")[TARGET_COL]
    .agg(["count", "sum", "mean"])
    .rename(columns={"sum": "DN_positive", "mean": "DN_rate"})
)

# Ensure no patient appears in more than one split.
assert split_map[ID_COL].is_unique


Target: primary_icd_dn_label

Split sizes:
split
train         32364
validation     6994
test           6954
Name: count, dtype: int64

DN label distribution by split:
            count  DN_positive   DN_rate
split                                   
test         6954          757  0.108858
train       32364         3836  0.118527
validation   6994          821  0.117386


In [4]:
# ============================================================
# 4. BUILD A LEAKAGE-CONSERVATIVE TABULAR DATASET
# ============================================================
"""
structured_features.parquet contains several useful variables, but some fields can
become leakage-prone depending on how they were extracted.

The default model below deliberately EXCLUDES:
- index_los_days            : known only after the admission ends
- n_eligible_admissions     : could include future utilization
- icu_total_los_days        : future information if predicting early
- full-admission mean_sbp / mean_dbp / glucose / urine output
- cohort max_creatinine / max_bun
- any DN outcome labels
- esrd_or_dialysis_flag

Instead, first-24h (or first-48h) glucose, creatinine, BUN and SBP are used.

Before publication, verify that comorbidity and medication indicators were also
constructed only from information available before your prediction cutoff.
"""

CATEGORICAL_COLS = [
    "gender",
    "race",
    "insurance",
    "admission_type",
]

NUMERIC_BASE_COLS = [
    "age_at_admission",
]

BINARY_COLS = [
    "type2_only_flag",
    "comorbid_cvd_ihd",
    "comorbid_heart_failure",
    "comorbid_hypertension",
    "comorbid_cerebrovascular",
    "comorbid_diabetic_neuropathy",
    "comorbid_infection_sepsis",
    "comorbid_copd",
    "comorbid_obesity",
    "comorbid_dyslipidemia",
    "med_insulin",
    "med_acei",
    "med_arb",
    "med_diuretic",
    "med_nephrotoxic_nsaid",
    "med_nephrotoxic_aminoglycoside",
    "med_nephrotoxic_vancomycin",
    "med_iodinated_contrast",
    "med_metformin",
]

if FEATURE_WINDOW == "24h":
    early = early24.copy()
    EARLY_NUMERIC_COLS = [
        "mean_glucose_24h",
        "max_creatinine_24h",
        "max_bun_24h",
        "mean_sbp_24h",
    ]
elif FEATURE_WINDOW == "48h":
    early = early48.copy()
    EARLY_NUMERIC_COLS = [
        "mean_glucose_48h",
        "max_creatinine_48h",
        "max_bun_48h",
        "mean_sbp_48h",
    ]
else:
    raise ValueError("FEATURE_WINDOW must be '24h' or '48h'.")

NUMERIC_COLS = NUMERIC_BASE_COLS + EARLY_NUMERIC_COLS

feature_cols = CATEGORICAL_COLS + NUMERIC_COLS + BINARY_COLS

missing_structured = [
    c for c in CATEGORICAL_COLS + NUMERIC_BASE_COLS + BINARY_COLS
    if c not in structured.columns
]
missing_early = [c for c in EARLY_NUMERIC_COLS if c not in early.columns]

if missing_structured:
    raise ValueError(f"Missing structured feature columns: {missing_structured}")
if missing_early:
    raise ValueError(f"Missing early feature columns: {missing_early}")

# Remove split from structured because patient_splits is authoritative.
structured_core = structured.drop(columns=["split"], errors="ignore").copy()

if structured_core[ID_COL].duplicated().any():
    raise ValueError(
        "structured_features.parquet has more than one row per patient. "
        "Do not silently aggregate it; review the extraction first."
    )

if early[ID_COL].duplicated().any():
    raise ValueError(
        f"early_features_{FEATURE_WINDOW}.parquet has more than one row per patient."
    )

tabular = (
    structured_core[
        [ID_COL, "index_hadm_id"] +
        CATEGORICAL_COLS +
        NUMERIC_BASE_COLS +
        BINARY_COLS
    ]
    .merge(early[[ID_COL] + EARLY_NUMERIC_COLS], on=ID_COL, how="left")
    .merge(
        label_base[[ID_COL, TARGET_COL, "split"]],
        on=ID_COL,
        how="inner",
    )
)

print("\nTabular modelling dataset:", tabular.shape)
print("Feature window:", FEATURE_WINDOW)


Tabular modelling dataset: (46312, 32)
Feature window: 24h


In [5]:
# ============================================================
# 5. TRAIN-ONLY IMPUTATION
# ============================================================
def fit_imputation(train_frame):
    medians = {}
    category_modes = {}

    for c in NUMERIC_COLS:
        x = pd.to_numeric(train_frame[c], errors="coerce")
        med = x.median()
        medians[c] = 0.0 if pd.isna(med) else float(med)

    for c in CATEGORICAL_COLS:
        values = train_frame[c].dropna().astype(str)
        category_modes[c] = values.mode().iloc[0] if len(values) else "MISSING"

    return medians, category_modes


def apply_imputation(frame, medians, category_modes):
    out = frame.copy()

    for c in NUMERIC_COLS:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(medians[c]).astype(float)

    for c in CATEGORICAL_COLS:
        out[c] = (
            out[c]
            .fillna(category_modes[c])
            .astype(str)
            .replace({"": "MISSING"})
        )

    for c in BINARY_COLS:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(0).astype(int)

    out[TARGET_COL] = out[TARGET_COL].astype(int)
    return out


train_tab_raw = tabular[tabular["split"] == "train"].copy()
val_tab_raw = tabular[tabular["split"] == "validation"].copy()
test_tab_raw = tabular[tabular["split"] == "test"].copy()

medians, category_modes = fit_imputation(train_tab_raw)

train_tab = apply_imputation(train_tab_raw, medians, category_modes)
val_tab = apply_imputation(val_tab_raw, medians, category_modes)
test_tab = apply_imputation(test_tab_raw, medians, category_modes)

print("\nTrain:", train_tab.shape)
print("Validation:", val_tab.shape)
print("Test:", test_tab.shape)


Train: (32364, 32)
Validation: (6994, 32)
Test: (6954, 32)


In [8]:
# ============================================================
# 6. COMMON EVALUATION FUNCTIONS
# ============================================================
def choose_threshold(y_true, y_prob):
    """
    Select the threshold on VALIDATION data only.
    The final test set remains untouched.
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).reshape(-1)

    thresholds = np.linspace(0.05, 0.95, 181)
    f1s = [
        f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        for t in thresholds
    ]
    return float(thresholds[int(np.argmax(f1s))])


def binary_metrics(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    return {
        "AUROC": roc_auc_score(y_true, y_prob),
        "AUPRC": average_precision_score(y_true, y_prob),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall_Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Brier": brier_score_loss(y_true, y_prob),
        "Threshold": threshold,
    }

In [9]:
# ============================================================
# 7. CREATE PRE-INDEX LONGITUDINAL ICD SEQUENCES
# ============================================================
"""
dx_sequences.parquet columns:
subject_id, hadm_id, visit_number, age_at_visit, days_since_prev_visit,
icd_code, icd_version, code_rank_within_visit, split

The index hospital admission is identified by index_hadm_id from cohort_labels.
For risk prediction, this code keeps ONLY visit_number < index_visit_number.
Thus the diagnosis sequence cannot directly contain the DN code assigned during
the outcome/index admission.
"""

dx_work = dx.copy()

required_dx = {
    ID_COL,
    "hadm_id",
    "visit_number",
    "icd_code",
    "icd_version",
    "code_rank_within_visit",
}
missing_dx = required_dx - set(dx_work.columns)
if missing_dx:
    raise ValueError(f"dx_sequences.parquet is missing: {missing_dx}")

index_ids = label_base[[ID_COL, "index_hadm_id"]].copy()

# Locate the sequence visit number corresponding to each patient's index admission.
index_visit = (
    dx_work.merge(index_ids, on=ID_COL, how="inner")
    .loc[lambda x: x["hadm_id"] == x["index_hadm_id"]]
    .groupby(ID_COL, as_index=False)["visit_number"]
    .min()
    .rename(columns={"visit_number": "index_visit_number"})
)

n_without_index_match = (
    label_base[ID_COL].nunique() - index_visit[ID_COL].nunique()
)

if n_without_index_match:
    warnings.warn(
        f"{n_without_index_match} patients do not have an index_hadm_id match "
        "inside dx_sequences.parquet. They will receive an empty prior-ICD sequence "
        "rather than risk including future visits."
    )

dx_pre = (
    dx_work
    .merge(index_visit, on=ID_COL, how="inner")
    .loc[lambda x: x["visit_number"] < x["index_visit_number"]]
    .copy()
)

# Distinguish ICD-9 and ICD-10 token namespaces.
dx_pre["icd_token"] = (
    "ICD" +
    dx_pre["icd_version"].astype(str) +
    "_" +
    dx_pre["icd_code"].astype(str)
)

dx_pre = dx_pre.sort_values(
    [ID_COL, "visit_number", "code_rank_within_visit"]
).reset_index(drop=True)

print("\nPre-index diagnosis rows:", len(dx_pre))
print("Patients with at least one prior diagnosis:", dx_pre[ID_COL].nunique())


Pre-index diagnosis rows: 0
Patients with at least one prior diagnosis: 0


C:\Users\oluwa\AppData\Local\Temp\ipykernel_32260\3916912671.py:45: UserWarning: 33 patients do not have an index_hadm_id match inside dx_sequences.parquet. They will receive an empty prior-ICD sequence rather than risk including future visits.
  warnings.warn(


In [10]:
# Match each patient's index admission to its visit number
index_check = (
    dx.merge(
        labels[["subject_id", "index_hadm_id"]],
        on="subject_id",
        how="inner"
    )
)

index_check = index_check[
    index_check["hadm_id"] == index_check["index_hadm_id"]
]

index_visit_numbers = (
    index_check
    .groupby("subject_id")["visit_number"]
    .min()
)

print("Patients matched:", len(index_visit_numbers))

print("\nIndex visit-number distribution:")
print(index_visit_numbers.value_counts().sort_index().head(20))

print(
    "\nPatients whose index admission is Visit 1:",
    (index_visit_numbers == 1).sum()
)

Patients matched: 46279

Index visit-number distribution:
visit_number
1    46279
Name: count, dtype: int64

Patients whose index admission is Visit 1: 46279


In [ ]:
# ============================================================
# 8. BUILD PATIENT SEQUENCE DICTIONARIES
# ============================================================
patient_sequence = defaultdict(list)

for row in dx_pre[
    [ID_COL, "visit_number", "code_rank_within_visit", "icd_token"]
].itertuples(index=False):
    patient_sequence[int(row.subject_id)].append(
        (
            int(row.visit_number),
            int(row.code_rank_within_visit),
            str(row.icd_token),
        )
    )


def prior_code_text(subject_id, max_codes=128):
    seq = patient_sequence.get(int(subject_id), [])
    if not seq:
        return "No prior diagnosis codes available."

    seq = seq[-max_codes:]
    return " ".join(token for _, _, token in seq)

In [ ]:
# ============================================================
# 9. CLINICALBERT EHR-TO-TEXT
# ============================================================
"""
IMPORTANT:
Bio_ClinicalBERT was pretrained on clinical notes, but no MIMIC-IV note file
was supplied here. This implementation serializes the SAFE structured variables
and pre-index diagnosis sequence into text.

For a paper, call this:
    "ClinicalBERT EHR-to-text adaptation"
rather than:
    "note-based ClinicalBERT".

If you later add MIMIC-IV-Note, replace build_ehr_text() with note text that
was documented before the prediction cutoff.
"""

def pretty_feature_name(name):
    return name.replace("_", " ")


def build_ehr_text(row):
    parts = [
        f"Age at admission {row['age_at_admission']}.",
        f"Gender {row['gender']}.",
        f"Race {row['race']}.",
        f"Insurance {row['insurance']}.",
        f"Admission type {row['admission_type']}.",
    ]

    for c in BINARY_COLS:
        if int(row[c]) == 1:
            parts.append(f"{pretty_feature_name(c)} present.")

    for c in EARLY_NUMERIC_COLS:
        value = row[c]
        parts.append(f"{pretty_feature_name(c)} {value:.3f}.")

    parts.append(
        "Prior diagnoses: " +
        prior_code_text(row[ID_COL], max_codes=128)
    )

    return " ".join(parts)


clinicalbert_frame = tabular[
    [ID_COL, TARGET_COL, "split"] + feature_cols
].copy()

clinicalbert_frame = apply_imputation(
    clinicalbert_frame,
    medians,
    category_modes,
)

clinicalbert_frame["clinical_text"] = clinicalbert_frame.apply(
    build_ehr_text,
    axis=1,
)

print("\nExample ClinicalBERT EHR-to-text record:")
print(clinicalbert_frame["clinical_text"].iloc[0][:1200])

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CLINICALBERT_NAME = "emilyalsentzer/Bio_ClinicalBERT"


class ClinicalBERTDataset(Dataset):
    def __init__(self, frame, tokenizer, max_len=256):
        self.texts = frame["clinical_text"].astype(str).tolist()
        self.labels = frame[TARGET_COL].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long,
        )
        return item


@torch.no_grad()
def predict_clinicalbert(model, loader):
    model.eval()
    y_all, p_all = [], []

    for batch in loader:
        y = batch.pop("labels").to(DEVICE)
        x = {k: v.to(DEVICE) for k, v in batch.items()}

        logits = model(**x).logits
        prob = torch.softmax(logits, dim=1)[:, 1]

        y_all.extend(y.cpu().numpy())
        p_all.extend(prob.cpu().numpy())

    return np.asarray(y_all), np.asarray(p_all)


def train_clinicalbert(
    frame,
    epochs=3,
    batch_size=None,
    lr=2e-5,
    max_len=CLINICALBERT_MAX_LEN,
    patience=2,
):
    if batch_size is None:
        batch_size = 8 if DEVICE.type == "cuda" else 2

    tr = frame[frame["split"] == "train"].reset_index(drop=True)
    va = frame[frame["split"] == "validation"].reset_index(drop=True)
    te = frame[frame["split"] == "test"].reset_index(drop=True)

    tokenizer = AutoTokenizer.from_pretrained(CLINICALBERT_NAME)

    model = AutoModelForSequenceClassification.from_pretrained(
        CLINICALBERT_NAME,
        num_labels=2,
    ).to(DEVICE)

    tr_loader = DataLoader(
        ClinicalBERTDataset(tr, tokenizer, max_len),
        batch_size=batch_size,
        shuffle=True,
    )
    va_loader = DataLoader(
        ClinicalBERTDataset(va, tokenizer, max_len),
        batch_size=batch_size,
        shuffle=False,
    )
    te_loader = DataLoader(
        ClinicalBERTDataset(te, tokenizer, max_len),
        batch_size=batch_size,
        shuffle=False,
    )

    y_train = tr[TARGET_COL].to_numpy()
    counts = np.bincount(y_train, minlength=2)
    weights = len(y_train) / (2 * np.maximum(counts, 1))
    weights = torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    )

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=0.01,
    )

    best_auc = -np.inf
    best_state = None
    no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for batch in tr_loader:
            labels_batch = batch.pop("labels").to(DEVICE)
            x = {k: v.to(DEVICE) for k, v in batch.items()}

            optimizer.zero_grad()
            logits = model(**x).logits

            loss = criterion(logits, labels_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )
            optimizer.step()

            running_loss += loss.item()

        y_val, p_val = predict_clinicalbert(model, va_loader)
        val_auc = roc_auc_score(y_val, p_val)

        print(
            f"ClinicalBERT epoch {epoch}: "
            f"loss={running_loss / max(len(tr_loader),1):.4f} "
            f"val_AUROC={val_auc:.4f}"
        )

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print("ClinicalBERT early stopping.")
                break

    model.load_state_dict(best_state)

    y_val, p_val = predict_clinicalbert(model, va_loader)
    threshold = choose_threshold(y_val, p_val)

    y_test, p_test = predict_clinicalbert(model, te_loader)
    metrics = binary_metrics(y_test, p_test, threshold)

    output_dir = DATA_DIR / "checkpoints" / "clinicalbert_ehr_to_text_dn"
    output_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    return model, metrics, p_test

In [ ]:
# ============================================================
# 10. MED-BERT-STYLE STRUCTURED EHR TRANSFORMER
# ============================================================
"""
This uses:
- ICD code embedding
- visit embedding
- code-rank/serialization embedding
- Transformer encoder

It is intentionally named Med-BERT-style because the original Med-BERT
pretrained weights are not being used here.
"""

PAD = "[PAD]"
UNK = "[UNK]"


def build_medbert_vocab(dx_frame, train_subject_ids, min_freq=1):
    train_set = set(int(x) for x in train_subject_ids)
    counts = Counter(
        dx_frame.loc[
            dx_frame[ID_COL].isin(train_set),
            "icd_token",
        ].astype(str)
    )

    vocab = {PAD: 0, UNK: 1}

    for token, n in sorted(counts.items()):
        if n >= min_freq:
            vocab[token] = len(vocab)

    return vocab


train_ids = label_base.loc[
    label_base["split"] == "train",
    ID_COL
].tolist()

MEDBERT_VOCAB = build_medbert_vocab(
    dx_pre,
    train_ids,
    min_freq=1,
)

print("Med-BERT-style vocabulary size:", len(MEDBERT_VOCAB))


class MedBERTDataset(Dataset):
    def __init__(
        self,
        patients,
        sequence_dict,
        vocab,
        max_len=256,
        max_visits=128,
        max_rank=128,
    ):
        self.patients = patients.reset_index(drop=True)
        self.sequence_dict = sequence_dict
        self.vocab = vocab
        self.max_len = max_len
        self.max_visits = max_visits
        self.max_rank = max_rank

    def __len__(self):
        return len(self.patients)

    def __getitem__(self, idx):
        row = self.patients.iloc[idx]
        sid = int(row[ID_COL])

        seq = self.sequence_dict.get(sid, [])

        # Keep the most recent diagnoses when sequence is very long.
        seq = seq[-self.max_len:]

        if len(seq) == 0:
            tokens = [self.vocab[UNK]]
            visit_ids = [0]
            rank_ids = [0]
        else:
            unique_visits = []
            visit_to_local = {}

            for visit_no, _, _ in seq:
                if visit_no not in visit_to_local:
                    visit_to_local[visit_no] = len(unique_visits)
                    unique_visits.append(visit_no)

            tokens = []
            visit_ids = []
            rank_ids = []

            for visit_no, rank, token in seq:
                tokens.append(
                    self.vocab.get(token, self.vocab[UNK])
                )
                visit_ids.append(
                    min(
                        visit_to_local[visit_no],
                        self.max_visits - 1,
                    )
                )
                rank_ids.append(
                    min(
                        max(int(rank) - 1, 0),
                        self.max_rank - 1,
                    )
                )

        length = len(tokens)
        pad_len = self.max_len - length

        tokens = tokens + [0] * pad_len
        visit_ids = visit_ids + [0] * pad_len
        rank_ids = rank_ids + [0] * pad_len
        mask = [1] * length + [0] * pad_len

        return {
            "input_ids": torch.tensor(tokens, dtype=torch.long),
            "visit_ids": torch.tensor(visit_ids, dtype=torch.long),
            "rank_ids": torch.tensor(rank_ids, dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.bool),
            "labels": torch.tensor(
                int(row[TARGET_COL]),
                dtype=torch.long,
            ),
        }


class MedBERTStyleClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        hidden_dim=192,
        n_heads=6,
        n_layers=6,
        ff_dim=768,
        dropout=0.1,
        max_visits=128,
        max_rank=128,
    ):
        super().__init__()

        self.code_embedding = nn.Embedding(
            vocab_size,
            hidden_dim,
            padding_idx=0,
        )
        self.visit_embedding = nn.Embedding(
            max_visits,
            hidden_dim,
        )
        self.rank_embedding = nn.Embedding(
            max_rank,
            hidden_dim,
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers,
        )

        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, 2)

    def forward(
        self,
        input_ids,
        visit_ids,
        rank_ids,
        attention_mask,
    ):
        x = (
            self.code_embedding(input_ids)
            + self.visit_embedding(visit_ids)
            + self.rank_embedding(rank_ids)
        )

        h = self.encoder(
            x,
            src_key_padding_mask=~attention_mask,
        )

        mask = attention_mask.unsqueeze(-1).float()

        pooled = (h * mask).sum(dim=1)
        pooled = pooled / mask.sum(dim=1).clamp(min=1.0)
        pooled = self.norm(pooled)

        return self.classifier(
            self.dropout(pooled)
        )


@torch.no_grad()
def predict_medbert(model, loader):
    model.eval()
    y_all, p_all = [], []

    for batch in loader:
        y = batch.pop("labels").to(DEVICE)
        x = {k: v.to(DEVICE) for k, v in batch.items()}

        logits = model(**x)
        prob = torch.softmax(logits, dim=1)[:, 1]

        y_all.extend(y.cpu().numpy())
        p_all.extend(prob.cpu().numpy())

    return np.asarray(y_all), np.asarray(p_all)


def train_medbert_style(
    labels_frame,
    epochs=20,
    batch_size=None,
    lr=1e-4,
    max_len=MEDBERT_MAX_LEN,
    patience=4,
):
    if batch_size is None:
        batch_size = 32 if DEVICE.type == "cuda" else 8

    tr = labels_frame[
        labels_frame["split"] == "train"
    ].reset_index(drop=True)

    va = labels_frame[
        labels_frame["split"] == "validation"
    ].reset_index(drop=True)

    te = labels_frame[
        labels_frame["split"] == "test"
    ].reset_index(drop=True)

    tr_loader = DataLoader(
        MedBERTDataset(
            tr,
            patient_sequence,
            MEDBERT_VOCAB,
            max_len=max_len,
        ),
        batch_size=batch_size,
        shuffle=True,
    )

    va_loader = DataLoader(
        MedBERTDataset(
            va,
            patient_sequence,
            MEDBERT_VOCAB,
            max_len=max_len,
        ),
        batch_size=batch_size,
        shuffle=False,
    )

    te_loader = DataLoader(
        MedBERTDataset(
            te,
            patient_sequence,
            MEDBERT_VOCAB,
            max_len=max_len,
        ),
        batch_size=batch_size,
        shuffle=False,
    )

    model = MedBERTStyleClassifier(
        vocab_size=len(MEDBERT_VOCAB)
    ).to(DEVICE)

    y_train = tr[TARGET_COL].to_numpy()
    counts = np.bincount(y_train, minlength=2)
    weights = len(y_train) / (2 * np.maximum(counts, 1))
    weights = torch.tensor(
        weights,
        dtype=torch.float32,
        device=DEVICE,
    )

    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=0.01,
    )

    best_auc = -np.inf
    best_state = None
    no_improve = 0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0

        for batch in tr_loader:
            labels_batch = batch.pop("labels").to(DEVICE)
            x = {k: v.to(DEVICE) for k, v in batch.items()}

            optimizer.zero_grad()

            logits = model(**x)
            loss = criterion(logits, labels_batch)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0,
            )

            optimizer.step()

            running_loss += loss.item()

        y_val, p_val = predict_medbert(
            model,
            va_loader,
        )

        val_auc = roc_auc_score(y_val, p_val)

        print(
            f"Med-BERT-style epoch {epoch}: "
            f"loss={running_loss / max(len(tr_loader),1):.4f} "
            f"val_AUROC={val_auc:.4f}"
        )

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = copy.deepcopy(
                model.state_dict()
            )
            no_improve = 0
        else:
            no_improve += 1

            if no_improve >= patience:
                print("Med-BERT-style early stopping.")
                break

    model.load_state_dict(best_state)

    y_val, p_val = predict_medbert(
        model,
        va_loader,
    )

    threshold = choose_threshold(
        y_val,
        p_val,
    )

    y_test, p_test = predict_medbert(
        model,
        te_loader,
    )

    metrics = binary_metrics(
        y_test,
        p_test,
        threshold,
    )

    output_dir = DATA_DIR / "checkpoints"
    output_dir.mkdir(parents=True, exist_ok=True)

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "vocab": MEDBERT_VOCAB,
            "target": TARGET_COL,
            "feature_window": FEATURE_WINDOW,
            "max_len": max_len,
        },
        output_dir / "medbert_style_dn.pt",
    )

    return model, metrics, p_test

In [ ]:
# ============================================================
# 11. TRANSTAB
# ============================================================
def train_transtab(
    train_frame,
    val_frame,
    test_frame,
    epochs=50,
    batch_size=None,
    lr=1e-4,
):
    import transtab

    if batch_size is None:
        batch_size = 128 if DEVICE.type == "cuda" else 32

    transtab.random_seed(SEED)

    X_train = train_frame[feature_cols].copy()
    y_train = train_frame[TARGET_COL].astype(int).copy()

    X_val = val_frame[feature_cols].copy()
    y_val = val_frame[TARGET_COL].astype(int).copy()

    X_test = test_frame[feature_cols].copy()
    y_test = test_frame[TARGET_COL].astype(int).to_numpy()

    model = transtab.build_classifier(
        CATEGORICAL_COLS,
        NUMERIC_COLS,
        BINARY_COLS,
    )

    output_dir = DATA_DIR / "checkpoints" / "transtab_dn"
    output_dir.mkdir(parents=True, exist_ok=True)

    training_arguments = {
        "num_epoch": epochs,
        "batch_size": batch_size,
        "lr": lr,
        "eval_metric": "val_loss",
        "eval_less_is_better": True,
        "output_dir": str(output_dir),
    }

    transtab.train(
        model,
        (X_train, y_train),
        (X_val, y_val),
        **training_arguments,
    )

    p_val = np.asarray(
        transtab.predict(model, X_val)
    ).reshape(-1)

    # Defensive conversion in case a version returns logits.
    if (
        np.nanmin(p_val) < 0
        or np.nanmax(p_val) > 1
    ):
        p_val = 1 / (1 + np.exp(-p_val))

    threshold = choose_threshold(
        y_val.to_numpy(),
        p_val,
    )

    p_test = np.asarray(
        transtab.predict(model, X_test)
    ).reshape(-1)

    if (
        np.nanmin(p_test) < 0
        or np.nanmax(p_test) > 1
    ):
        p_test = 1 / (1 + np.exp(-p_test))

    metrics = binary_metrics(
        y_test,
        p_test,
        threshold,
    )

    return model, metrics, p_test

In [ ]:
# ============================================================
# 12. FT-TRANSFORMER
# ============================================================
def find_positive_probability_column(prediction_df):
    preferred = [
        f"{TARGET_COL}_1_probability",
        "1_probability",
    ]

    for c in preferred:
        if c in prediction_df.columns:
            return c

    probability_cols = [
        c for c in prediction_df.columns
        if "probability" in str(c).lower()
    ]

    class1 = [
        c for c in probability_cols
        if (
            str(c).lower().endswith("1_probability")
            or "_1_" in str(c).lower()
        )
    ]

    if len(class1) == 1:
        return class1[0]

    raise ValueError(
        "Could not identify the positive-class probability column. "
        f"Prediction columns: {prediction_df.columns.tolist()}"
    )


def train_fttransformer(
    train_frame,
    val_frame,
    test_frame,
    epochs=50,
    batch_size=None,
    lr=1e-3,
):
    from pytorch_tabular import TabularModel
    from pytorch_tabular.config import (
        DataConfig,
        OptimizerConfig,
        TrainerConfig,
    )
    from pytorch_tabular.models import FTTransformerConfig

    if batch_size is None:
        batch_size = 256 if DEVICE.type == "cuda" else 64

    # PyTorch Tabular treats binary variables well as categorical features.
    ft_cat_cols = CATEGORICAL_COLS + BINARY_COLS

    keep_cols = (
        NUMERIC_COLS
        + ft_cat_cols
        + [TARGET_COL]
    )

    tr = train_frame[keep_cols].copy()
    va = val_frame[keep_cols].copy()
    te = test_frame[keep_cols].copy()

    data_config = DataConfig(
        target=[TARGET_COL],
        continuous_cols=NUMERIC_COLS,
        categorical_cols=ft_cat_cols,
        normalize_continuous_features=True,
    )

    checkpoint_path = (
        DATA_DIR /
        "checkpoints" /
        "fttransformer_dn"
    )
    checkpoint_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    trainer_config = TrainerConfig(
        batch_size=batch_size,
        max_epochs=epochs,
        early_stopping="valid_loss",
        early_stopping_mode="min",
        early_stopping_patience=5,
        checkpoints="valid_loss",
        checkpoints_path=str(checkpoint_path),
        load_best=True,
        accelerator="auto",
    )

    optimizer_config = OptimizerConfig()

    model_config = FTTransformerConfig(
        task="classification",
        input_embed_dim=32,
        num_attn_blocks=3,
        num_heads=4,
        learning_rate=lr,
    )

    tabular_model = TabularModel(
        data_config=data_config,
        model_config=model_config,
        optimizer_config=optimizer_config,
        trainer_config=trainer_config,
        verbose=True,
    )

    tabular_model.fit(
        train=tr,
        validation=va,
    )

    val_pred = tabular_model.predict(
        va,
        include_input_features=False,
    )

    test_pred = tabular_model.predict(
        te,
        include_input_features=False,
    )

    prob_col = find_positive_probability_column(
        val_pred
    )

    p_val = val_pred[prob_col].to_numpy(
        dtype=float
    )

    threshold = choose_threshold(
        va[TARGET_COL].to_numpy(),
        p_val,
    )

    p_test = test_pred[prob_col].to_numpy(
        dtype=float
    )

    metrics = binary_metrics(
        te[TARGET_COL].to_numpy(),
        p_test,
        threshold,
    )

    return tabular_model, metrics, p_test

In [ ]:
# ============================================================
# 13. TRAIN ALL MODELS
# ============================================================
results = []

predictions = test_tab[
    [ID_COL, TARGET_COL]
].copy()

predictions = predictions.rename(
    columns={TARGET_COL: "y_true"}
)

if RUN_CLINICALBERT:
    print("\n" + "=" * 70)
    print("TRAINING CLINICALBERT EHR-TO-TEXT")
    print("=" * 70)

    (
        clinicalbert_model,
        clinicalbert_metrics,
        p_clinicalbert,
    ) = train_clinicalbert(
        clinicalbert_frame
    )

    results.append(
        {
            "Model": "ClinicalBERT EHR-to-text",
            **clinicalbert_metrics,
        }
    )

    predictions[
        "ClinicalBERT_probability"
    ] = p_clinicalbert


if RUN_MEDBERT:
    print("\n" + "=" * 70)
    print("TRAINING MED-BERT-STYLE")
    print("=" * 70)

    (
        medbert_model,
        medbert_metrics,
        p_medbert,
    ) = train_medbert_style(
        label_base
    )

    results.append(
        {
            "Model": "Med-BERT-style",
            **medbert_metrics,
        }
    )

    predictions[
        "MedBERT_style_probability"
    ] = p_medbert


if RUN_TRANSTAB:
    print("\n" + "=" * 70)
    print("TRAINING TRANSTAB")
    print("=" * 70)

    (
        transtab_model,
        transtab_metrics,
        p_transtab,
    ) = train_transtab(
        train_tab,
        val_tab,
        test_tab,
    )

    results.append(
        {
            "Model": "TransTab",
            **transtab_metrics,
        }
    )

    predictions[
        "TransTab_probability"
    ] = p_transtab


if RUN_FTTRANSFORMER:
    print("\n" + "=" * 70)
    print("TRAINING FT-TRANSFORMER")
    print("=" * 70)

    (
        ft_model,
        ft_metrics,
        p_ft,
    ) = train_fttransformer(
        train_tab,
        val_tab,
        test_tab,
    )

    results.append(
        {
            "Model": "FT-Transformer",
            **ft_metrics,
        }
    )

    predictions[
        "FTTransformer_probability"
    ] = p_ft

In [ ]:
# ============================================================
# 14. COMPARE AND SAVE RESULTS
# ============================================================
results_df = pd.DataFrame(results)

if not results_df.empty:
    results_df = results_df.sort_values(
        "AUROC",
        ascending=False,
    ).reset_index(drop=True)

    print("\nFINAL MODEL COMPARISON")
    print(results_df.to_string(index=False))

    results_path = DATA_DIR / (
        f"dn_model_comparison_{TARGET_COL}_{FEATURE_WINDOW}.csv"
    )

    results_df.to_csv(
        results_path,
        index=False,
    )

predictions_path = DATA_DIR / (
    f"dn_test_predictions_{TARGET_COL}_{FEATURE_WINDOW}.csv"
)

predictions.to_csv(
    predictions_path,
    index=False,
)

config_path = DATA_DIR / (
    f"dn_experiment_config_{TARGET_COL}_{FEATURE_WINDOW}.json"
)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "target": TARGET_COL,
            "feature_window": FEATURE_WINDOW,
            "categorical_cols": CATEGORICAL_COLS,
            "numeric_cols": NUMERIC_COLS,
            "binary_cols": BINARY_COLS,
            "seed": SEED,
            "device": str(DEVICE),
        },
        f,
        indent=2,
    )

print("\nSaved:")
if not results_df.empty:
    print(results_path)
print(predictions_path)
print(config_path)

In [ ]:
# ============================================================
# 15. OPTIONAL: REPEAT WITH ALTERNATIVE DN DEFINITIONS
# ============================================================
"""
For a sensitivity analysis, rerun the notebook after changing:

TARGET_COL = "secondary_lab_proxy_label"

and then:

TARGET_COL = "strict_sensitivity_label"

For the early-window sensitivity analysis, change:

FEATURE_WINDOW = "48h"

Keep patient_splits.parquet unchanged so all model comparisons use the same
patients in train, validation and test sets.
"""